# 02 · Data quality monitor

A read-only health report on the curated layer: pipeline runs, rejected rows, completeness, freshness
and the recommender model. It never calls the TMDB API and never changes any data.

Run it by hand, or let `python -m pipeline.report` execute it with papermill and save a dated copy in
`notebooks/reports/`. The last section is a pass / warn / fail checklist.

In [ ]:
# Parameters (papermill overrides these)
REPORT_DATE = None           # YYYY-MM-DD shown in the report (None means today)
RECENT_RUNS = 30             # how many pipeline runs to chart
MAX_DAYS_SINCE_SUCCESS = 8   # warn if the last successful run is older than this
MAX_DATA_AGE_DAYS = 45       # movies not re-fetched for this long count as stale
MIN_POSTER_COVERAGE = 0.95   # share of movies that should have a poster
MIN_TRAILER_COVERAGE = 0.85  # share of movies that should have a trailer

In [ ]:
import sys
from pathlib import Path

# Find the project root (the folder that contains pipeline/config.py) so the notebook
# works from Jupyter, from papermill and from Task Scheduler alike.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pipeline" / "config.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json
import sqlite3
from contextlib import closing
from datetime import date, datetime

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from pipeline import config, recommender

REPORT_DATE = REPORT_DATE or date.today().isoformat()
today = date.fromisoformat(REPORT_DATE)

# Chart style: one blue for single-series charts, blue + orange when two series are compared.
BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "figure.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#c3c2b7",
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.labelcolor": MUTED, "axes.labelsize": 9, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "legend.frameon": False, "legend.fontsize": 9, "font.family": "sans-serif",
})


def hbar(series, title, xlabel, fmt="{:,.0f}", color=BLUE):
    """Horizontal bar chart with a value label on every bar (largest at the top)."""
    series = series.sort_values()
    fig, ax = plt.subplots(figsize=(7.5, 0.38 * len(series) + 1.1))
    bars = ax.barh(series.index.astype(str), series.values, color=color, height=0.62)
    ax.bar_label(bars, labels=[fmt.format(v) for v in series.values], padding=4, fontsize=9, color=INK)
    ax.set(title=title, xlabel=xlabel)
    ax.grid(axis="y", visible=False)
    ax.margins(x=0.12)
    plt.show()


with closing(sqlite3.connect(config.DB_PATH)) as conn:
    movies = pd.read_sql("SELECT * FROM movies", conn)
    genres = pd.read_sql("SELECT * FROM genres", conn)
    runs = pd.read_sql("SELECT * FROM pipeline_runs ORDER BY run_id", conn)
    counts = {t: conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
              for t in ("movies", "genres", "cast_crew", "trailers", "pipeline_runs")}
    orphans = {t: conn.execute(
        f"SELECT COUNT(*) FROM {t} WHERE movie_id NOT IN (SELECT movie_id FROM movies)").fetchone()[0]
        for t in ("genres", "cast_crew", "trailers")}

artifact = recommender.load_neighbours()
model_report = json.loads(config.EVAL_REPORT.read_text("utf-8")) if config.EVAL_REPORT.exists() else None
print(f"Report date {REPORT_DATE} | database {config.DB_PATH}")

## 1. Summary

In [ ]:
ok_runs = runs[runs["status"] == "success"]
last_ok = ok_runs["finished_at"].max() if len(ok_runs) else None
summary = {
    "Movies": f"{counts['movies']:,}",
    "Genre rows": f"{counts['genres']:,}",
    "Cast and crew rows": f"{counts['cast_crew']:,}",
    "Trailers": f"{counts['trailers']:,}",
    "Pipeline runs recorded": f"{counts['pipeline_runs']:,} ({(runs['status'] == 'failed').sum()} failed)",
    "Last successful run": last_ok or "never",
    "Model built": artifact["built_at"] if artifact else "not built yet",
    "Model in use": artifact["default_method"] if artifact else "-",
}
pd.Series(summary, name=f"as of {REPORT_DATE}").to_frame()

## 2. Pipeline runs

Each bar is one run that loaded data: movies new to the database versus existing movies that were refreshed.

In [ ]:
recent = runs.tail(RECENT_RUNS)
display(recent[["run_id", "started_at", "stage", "status", "duration_seconds", "rows_extracted",
                "rows_rejected", "rows_inserted", "rows_updated", "model_refreshed"]]
        .sort_values("run_id", ascending=False).head(10).reset_index(drop=True))

loaded = recent[recent["rows_loaded"] > 0]
if loaded.empty:
    print("No runs have loaded data yet.")
else:
    labels = [f"#{i}\n{d[5:10]}" for i, d in zip(loaded["run_id"], loaded["started_at"])]
    fig, ax = plt.subplots(figsize=(max(5, 0.9 * len(loaded) + 2.5), 3.4))
    new = ax.bar(labels, loaded["rows_inserted"], color=BLUE, width=0.55, label="New movies")
    upd = ax.bar(labels, loaded["rows_updated"], bottom=loaded["rows_inserted"], color=ORANGE, width=0.55,
                 label="Updated movies", edgecolor="#fcfcfb", linewidth=1.5)
    ax.bar_label(upd, labels=[f"{v:,}" for v in loaded["rows_loaded"]], padding=3, fontsize=9, color=INK)
    ax.set(title="Movies loaded per run", xlabel="Run (id and date)", ylabel="Movies")
    ax.grid(axis="x", visible=False)
    ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
    ax.margins(y=0.15)
    plt.show()

In [ ]:
timed = recent.dropna(subset=["duration_seconds"])
if len(timed):
    labels = [f"#{i} {s}" for i, s in zip(timed["run_id"], timed["stage"])]
    hbar(pd.Series(timed["duration_seconds"].values / 60, index=labels),
         "Run duration (longest at the top)", "Minutes", fmt="{:.1f}")

failed = runs[runs["status"] != "success"][["run_id", "started_at", "stage", "status", "error_message"]]
display(Markdown("**Runs that did not succeed**")) if len(failed) else print("Every recorded run succeeded.")
if len(failed):
    display(failed.reset_index(drop=True))

## 3. Validation and quarantine

Rows that fail a rule are written to `data/quarantine/` instead of stopping the pipeline. Malformed poster paths or trailer keys are repaired to empty and counted as warnings.

In [ ]:
manifests = []
for path in sorted(config.STAGED_DIR.glob("*/_manifest.json")):
    m = json.loads(path.read_text("utf-8"))
    manifests.append({"run_date": m["run_date"], "rows_in": m["rows_in"], "rows_valid": m["rows_valid"],
                      "rows_rejected": m["rows_rejected"], **{f"rejected: {k}": v for k, v in m["rejection_reasons"].items()},
                      **{f"warning: {k}": v for k, v in m["warnings"].items()}})
if manifests:
    by_date = pd.DataFrame(manifests).set_index("run_date").fillna(0).astype(int)
    by_date["rejected %"] = (100 * by_date["rows_rejected"] / by_date["rows_in"].clip(lower=1)).round(2)
    display(by_date.T.map(lambda v: f"{v:g}"))
else:
    print("No staged manifests found (the staged layer is not kept in git).")

rejected_files = sorted(config.QUARANTINE_DIR.glob("*/rejected_movies.parquet"))
if rejected_files:
    quarantine = pd.concat([pd.read_parquet(p) for p in rejected_files], ignore_index=True)
    reasons = quarantine["reasons"].str.split("; ").explode().value_counts()
    hbar(reasons, "Quarantined rows by reason (all run dates)", "Rows")
    display(Markdown("**Sample of quarantined rows**"))
    display(quarantine[[c for c in ("run_date", "movie_id", "title", "reasons") if c in quarantine]].tail(8)
            .reset_index(drop=True))
else:
    print("Nothing is in quarantine.")

## 4. Completeness

Share of movies with no value in each field. Title, overview and rating can never be missing because validation rejects those rows.

In [ ]:
def is_empty(col):
    s = movies[col]
    return s.isna() | s.astype(str).str.strip().isin(["", "[]"])

fields = {"poster_path": "Poster", "trailer_key": "Trailer", "director": "Director", "top_cast": "Cast",
          "keywords": "Keywords", "genres": "Genres", "tagline": "Tagline", "release_date": "Release date",
          "runtime": "Runtime", "imdb_id": "IMDb id"}
missing = pd.Series({label: is_empty(col).mean() * 100 for col, label in fields.items()})
hbar(missing, f"Missing values by field ({len(movies):,} movies)", "% of movies", fmt="{:.1f}%")

unrated = (movies["vote_count"].fillna(0) == 0).sum()
print(f"{unrated:,} movies have no votes yet (mostly unreleased); the app shows them as 'Not yet rated'.")

## 5. Distributions

A quick sanity check that the catalogue looks like a real film collection.

In [ ]:
rated = movies[movies["vote_count"].fillna(0) > 0]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.3))
axes[0].hist(rated["vote_average"], bins=[x / 2 for x in range(0, 21)], color=BLUE, edgecolor="#fcfcfb", linewidth=1)
axes[0].set(title=f"Rating (movies with votes, median {rated['vote_average'].median():.1f})",
            xlabel="TMDB rating (0-10)", ylabel="Movies")
decades = (movies["release_year"].dropna().astype(int) // 10 * 10).value_counts().sort_index()
axes[1].bar([f"{d}s" for d in decades.index], decades.values, color=BLUE, width=0.62)
axes[1].set(title="Movies by release decade", xlabel="Decade", ylabel="Movies")
axes[1].tick_params(axis="x", rotation=45)
for ax in axes:
    ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

hbar(genres["genre_name"].value_counts(), "Movies per genre (a movie can have several)", "Movies")

## 6. Freshness

When each movie was last fetched from TMDB. The pipeline re-fetches a movie when TMDB reports a change, or after 30 days.

In [ ]:
fetched = pd.to_datetime(movies["extracted_at"]).dt.date
age_days = fetched.map(lambda d: (today - d).days)
by_day = fetched.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(max(5, 0.8 * len(by_day) + 2), 3))
bars = ax.bar([d.isoformat()[5:] for d in by_day.index], by_day.values, color=BLUE, width=0.55)
ax.bar_label(bars, labels=[f"{v:,}" for v in by_day.values], padding=3, fontsize=9, color=INK)
ax.set(title="Movies by date last fetched", xlabel="Fetch date (month-day)", ylabel="Movies")
ax.grid(axis="x", visible=False)
ax.margins(y=0.15)
plt.show()
stale = int((age_days > MAX_DATA_AGE_DAYS).sum())
print(f"Oldest data: {age_days.max()} days | median: {int(age_days.median())} days | "
      f"older than {MAX_DATA_AGE_DAYS} days: {stale:,} movies")

## 7. Recommender model

Both methods are scored against TMDB's own recommendations for the most-voted movies. TMDB's list is based on user behaviour, so a content-only model is not expected to score near 1.0; the comparison between the two methods is what matters.

In [ ]:
if not (artifact and model_report and model_report["metrics"]):
    print("No model comparison available yet. Run: python -m pipeline.refresh_model")
else:
    metrics = pd.DataFrame(model_report["metrics"]).rename(
        columns={"tfidf": "TF-IDF", "embeddings": "Embeddings"},
        index={"precision@10": "Precision@10", "recall@10": "Recall@10", "hit_rate": "Hit rate"})
    fig, ax = plt.subplots(figsize=(7, 3.3))
    x = range(len(metrics))
    for offset, (name, color) in zip((-0.2, 0.2), (("TF-IDF", BLUE), ("Embeddings", ORANGE))):
        bars = ax.bar([i + offset for i in x], metrics[name], width=0.36, color=color, label=name)
        ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=9, color=INK)
    ax.set_xticks(list(x), metrics.index)
    ax.set(title=f"Overlap with TMDB recommendations ({model_report['evaluated_movies']} movies)",
           ylabel="Score (0-1)", ylim=(0, 1))
    ax.grid(axis="x", visible=False)
    ax.legend(loc="upper left")
    plt.show()
    print(f"Winner: {model_report['winner']} | built {model_report['built_at']} | {model_report['n_movies']:,} movies")
    for title, lists in model_report["spot_checks"].items():
        display(Markdown(f"**{title}**"))
        display(pd.DataFrame({"TF-IDF": lists["tfidf"][:5], "Embeddings": lists["embeddings"][:5]},
                             index=range(1, 6)))

## 8. Automated checks

`PASS` is healthy, `WARN` deserves a look, `FAIL` means the data or the pipeline needs fixing. The report itself never stops on a failed check.

In [ ]:
checks = []

def check(name, ok, detail, level="FAIL"):
    checks.append({"Check": name, "Result": "PASS" if ok else level, "Detail": detail})

check("Database has movies", len(movies) > 0, f"{len(movies):,} movies")
dupes = int(movies["movie_id"].duplicated().sum())
check("No duplicate movie IDs", dupes == 0, f"{dupes} duplicates")
blank = int(is_empty("title").sum() + is_empty("overview").sum())
check("No empty titles or overviews", blank == 0, f"{blank} empty values")
bad_rating = int((~movies["vote_average"].between(config.RATING_MIN, config.RATING_MAX)).sum())
check("Ratings are between 0 and 10", bad_rating == 0, f"{bad_rating} out of range")
check("No orphan child rows", sum(orphans.values()) == 0, f"{orphans}")

poster = 1 - is_empty("poster_path").mean()
check("Poster coverage", poster >= MIN_POSTER_COVERAGE, f"{poster:.1%} (target {MIN_POSTER_COVERAGE:.0%})", "WARN")
trailer = 1 - is_empty("trailer_key").mean()
check("Trailer coverage", trailer >= MIN_TRAILER_COVERAGE, f"{trailer:.1%} (target {MIN_TRAILER_COVERAGE:.0%})", "WARN")
check("Data is fresh", stale == 0, f"{stale:,} movies older than {MAX_DATA_AGE_DAYS} days", "WARN")

if len(runs):
    last = runs.iloc[-1]
    check("Latest run succeeded", last["status"] == "success", f"run #{last['run_id']}: {last['status']}")
if last_ok:
    days = (today - datetime.fromisoformat(last_ok).date()).days
    check("A run succeeded recently", days <= MAX_DAYS_SINCE_SUCCESS,
          f"{days} days ago (limit {MAX_DAYS_SINCE_SUCCESS})", "WARN")
else:
    check("A run succeeded recently", False, "no successful run recorded")

in_sync = artifact is not None and set(artifact["movie_ids"].tolist()) == set(movies["movie_id"])
check("Model matches the database", in_sync,
      "same movies in both" if in_sync else "model is missing or out of date - run the refresh stage")

results = pd.DataFrame(checks)
tally = results["Result"].value_counts()
display(Markdown(f"**{tally.get('PASS', 0)} passed · {tally.get('WARN', 0)} warnings · {tally.get('FAIL', 0)} failed**"))
shade = {"PASS": "background-color: #d9f2d9", "WARN": "background-color: #fde9b8", "FAIL": "background-color: #f6c9c9"}
results.style.map(lambda v: shade.get(v, "") + "; color: #0b0b0b; font-weight: 600", subset=["Result"]).hide(axis="index")